<a href="https://colab.research.google.com/github/cbadenes/curso-pln/blob/main/notebooks/05_Transformers_con_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 1 - Listado 1 del tema 5

Se van a probar los transformers para análisis de sentimientos en español. 


In [1]:
# Import necesarios
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, MultiHeadAttention, LayerNormalization, Embedding, GlobalAveragePooling1D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Datos de ejemplo. En este primer ejercicio se utilizan frases simples para probar los transformers y familiarizarse con ellos. 
sentences = [
    'Estoy un poco harto del día a día, nada mejora',
    'Hoy es un buen día',
    'No se te ve satisfecho con el trabajo',
    'Este paisaje es hermoso y bonito'
]
labels = [0, 1, 0, 1]  # 1: Positivo, 0: Negativo

## Preprocesamiento del Texto

Convertimos las palabras a números usando un tokenizador

In [2]:
# Preparar el texto
vocab_size = 1000
max_length = 10

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
X = pad_sequences(sequences, maxlen=max_length)

# Convertir a tensores de TensorFlow
X = tf.convert_to_tensor(X, dtype=tf.float32)
labels = tf.convert_to_tensor(labels, dtype=tf.float32)

## Creacion del modelo Transformer

El modelo tiene las siguientes capas clave:
1. Embedding: Convierte palabras en vectores
2. Multi-Head Attention: La "magia" del Transformer que permite procesar relaciones entre palabras
3. Layer Normalization: Ayuda al entrenamiento
4. Dense: Capa final para clasificación

In [3]:
# Crear modelo Transformer simplificado
def create_transformer_classifier(vocab_size, max_length):
    inputs = Input(shape=(max_length,))

    # Capa de Embedding
    embedding_layer = tf.keras.layers.Embedding(vocab_size, 32)(inputs)   # <--------- EN VEZ DE 32 NO SERÍA MAX_LENGTH???

    # Multi-Head Attention (la parte mágica del Transformer)
    attention = MultiHeadAttention(num_heads=2, key_dim=32)(embedding_layer, embedding_layer, embedding_layer)  # < ------PORQUE SE PONE EMBEDDING LAYER 3 VECES?

    # Normalización y Residual connection
    x = LayerNormalization()(attention + embedding_layer)

    # Clasificación final
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs)

# Crear y compilar modelo
model = create_transformer_classifier(vocab_size, max_length)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Resumen del modelo
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 10, 32)    │     32,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 32)    │      8,416 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0],  │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 32)    │          0 │ multi_head_atten… │
│                     │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 32)    │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         33 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,513 (158.25 KB)

 Trainable params: 40,513 (158.25 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamiento y Ejecución

Entrenamos el modelo y probamos con nuevas frases

In [10]:
# Entrenar modelo
history = model.fit(
    X, labels,
    epochs=10,
    batch_size=2,
    verbose=1
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.6667 - loss: 0.7316
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5000 - loss: 0.5482
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 0.4157
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 0.3556
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.2410
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.1954
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 1.0000 - loss: 0.1434
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.1196
Epoch 9/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.0744
Epoch 10/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.0608


In [11]:
# Probar con nuevas frases
#test_sentences = [
 #   "Este producto es excelente",
  #  "Odio este servicio",
#]

test_sentences = [
    'No fui al estreno de la película porque nadie me quería acompañar',
    'Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio',
    'Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor',
    'Al final decidí no ir al cine porque estaba cansada'
]

# Preparar texto de prueba
test_sequences = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_sequences, maxlen=max_length)
test_padded = tf.convert_to_tensor(test_padded, dtype=tf.float32)

# Realizar predicciones
predictions = model.predict(test_padded)

# Mostrar resultados
for sentence, prediction in zip(test_sentences, predictions):
    sentiment = "Positivo" if prediction > 0.5 else "Negativo"
    print(f"Frase: '{sentence}'")
    print(f"Sentimiento: {sentiment} (probabilidad: {prediction[0]:.2f})")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 556ms/step
Frase: 'No fui al estreno de la película porque nadie me quería acompañar'
Sentimiento: Negativo (probabilidad: 0.01)

Frase: 'Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio'
Sentimiento: Negativo (probabilidad: 0.01)

Frase: 'Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor'
Sentimiento: Negativo (probabilidad: 0.47)

Frase: 'Al final decidí no ir al cine porque estaba cansada'
Sentimiento: Negativo (probabilidad: 0.01)

